In [96]:
import torch
import random
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [97]:
words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))

s_i = {s:i+1 for i, s in enumerate(chars)}
s_i["."] = 0

i_s = {i:s for s, i in s_i.items()}

# building dataset

block_size = 3

def build_dataset(words):
    
    X, Y = [], []
    for w in words:
        # print(w)
        context = [0] * block_size
        
        for ch in w + ".":
            ix = s_i[ch]
            X.append(context)
            Y.append(ix)
            # print("".join(i_s[i] for i in context), "---->", i_s[ix])
            context = context[1:] + [ix]
            
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

In [98]:
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [ ]:
n_embd = 10 # dimensionality of character embedding vector
n_chars = 27
n_hidden = 200 # number of neurons in hidden layer

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((n_chars, n_embd), generator=g)

w1 = torch.randn((n_embd * block_size, n_hidden)) * (5/3) / (30 ** 0.5) # kaiming initialization
# b1 = torch.randn(n_hidden) * 0.01

w2 = torch.randn((n_hidden, n_chars)) * 0.01
b2 = torch.randn(n_chars) * 0.1

bngain = torch.ones((1, n_hidden)) # * 0.1
bnbias = torch.zeros((1, n_hidden)) # * 0.1

parameters = [C, w1, w2, b2, bngain, bnbias]

for p in parameters:
    p.requires_grad = True

In [100]:
lre = torch.linspace(-3, 0, 1000)
lrs = 10 ** lre

lri = []

def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    
    print(f"{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}")

In [141]:
# training loop

max_steps = 200000

batch_size = 32
n = batch_size

lossi = []
stepi = []

ix = torch.randint(0, Xtr.shape[0], (batch_size,))
Xb, Yb = Xtr[ix], Ytr[ix]

# forward pass

# linear layer
# --------------------------------------
emb = C[Xb]
embcat = emb.view(-1, 30)
hprebn = embcat @ w1 # + b1
# --------------------------------------

# batch normalization layer **
# --------------------------------------
bnmeani = 1 / n * hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff ** 2
bnvar = 1 / (n - 1) * (bndiff2).sum(0, keepdim=True)
bnvar_inv = (bnvar + 1e-5) ** -0.5
bnraw = bndiff * bnvar_inv

hpreact = bngain * bnraw + bnbias # added an epsilon to bnstdi
# --------------------------------------

# non-linear layer
# --------------------------------------
h = torch.tanh(hpreact)

logits = h @ w2 + b2
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum ** -1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()
# --------------------------------------

# backward pass
for p in parameters:
    p.grad = None
    
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv,
          norm_logits, logit_maxes, logits, h, hpreact, bnmeani, 
          embcat, emb, bnstdi, bngain]:
    t.retain_grad()

loss.backward()
loss

# update
# lr = lrs[i]
# lr = 0.01
# for p in parameters:
#     p.data += -lr * p.grad
    
# track stats  
# lri.append(lre[i])
#if i % 1000 == 0:
#    print(f"{i:7d}/{max_steps:7d}: {loss.item():4f}")

#stepi.append(i)
#lossi.append(loss.log10().item())

tensor(3.2925, grad_fn=<NegBackward0>)

In [102]:
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [143]:
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1 / n
dprobs = (1.0 / probs) * dlogprobs
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True) # *
dcounts = counts_sum_inv * dprobs # *
dcounts_sum = (-counts_sum ** -2) * dcounts_sum_inv
dcounts += torch.ones_like(counts) * dcounts_sum

dnorm_logits = dcounts * counts
dlogits = dnorm_logits.clone()
dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True) # *
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes # *
dh = dlogits @ w2.T # *
dw2 = h.T @ dlogits # *
db2 = dlogits.sum(0) # *

dhpreact = (1.0 - (h ** 2)) * dh # *

dbngain = dhpreact * bnraw
dbnraw = dhpreact * dbngain
dbnbias = dhpreact.clone()

dbndiff = dbnraw * bnvar_inv
dbnvar_inv = dbnraw * bndiff
dhprebn = dbndiff.clone()
dbnmeani = -dbndiff

dbnvar = -0.5 * (bnvar_inv ** -1.5) * dbnvar_inv
dbndiff2 = torch.ones_like(bndiff2) * (1 / (n - 1)) * dbnvar
dbndiff += 2 * bndiff
dhprebn += torch.ones_like(hprebn) * (1 / n) * dbnmeani


# dbngain = (hpreact - bnmeani) * ((1 / ((bnstdi + 0.001) + bnbias)) * dhpreact) # *
# dbnmeani = -((1 / ((bnstdi + 0.001) + bnbias)) * dhpreact) # *

# dhpreactbnmeani = bngain * dbngainhpreactbnmeani # *
# dhpreact += dhpreactbnmeani # *


# dbnstdi01 = (bngain * (hpreact - bnmeani)) * dhpreact # *
# dbnbias = dhpreact.clone() # *
# dbnstdi01 = dbnstdi01.clone() # *
# dbnstdi = dbnstdi01.clone() # *

# dhpreact += dbnmeani * (1 / n)
# dhpreact +=

# dbnbias = dhpreact.clone()
# dbnstdi = hpreact.clone()
# dbngain = (hpreact - bnmeani) * (1 / ((bnstdi + 0.001) + bnbias)) * dhpreact
# dhpreact += bngain * (1 / ((bnstdi + 0.001) + bnbias)) * dhpreact
# dbnmeani = -(bngain * (1 / ((bnstdi + 0.001) + bnbias)) * dhpreact)

# dhpreact += dbnstdi * # backprop thru std
# dhpreact += (1 / n) * dbnmeani

dembcat = dhprebn @ w1.T
dw1 = embcat.T @ dhprebn

dC = torch.zeros_like(C)
dC[Xb] = dembcat.view(32, 3, 10)

cmp("logprobs", dlogprobs, logprobs)
cmp("probs", dprobs, probs)
cmp("counts_sum_inv", dcounts_sum_inv, counts_sum_inv)
cmp("counts_sum", dcounts_sum, counts_sum)
cmp("counts", dcounts, counts)
cmp("norm_logits", dnorm_logits, norm_logits)
cmp("norm_logits", dnorm_logits, norm_logits)
cmp("logits", dlogits, logits)
cmp("logit_maxes", dlogit_maxes, logit_maxes)
cmp("dh", dh, h)
cmp("dw2", dw2, w2)
cmp("db2", db2, b2)
cmp("dhpreact", dhpreact, hpreact)
cmp("dbnmeani", dbnmeani, bnmeani)
cmp("dbnstdi", dbnstdi, bnstdi)
cmp("dbngain", dbngain, bngain)
cmp("dembact", dembcat, embcat)
cmp("dw1", dw1, w1)
cmp("dC", dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
dh              | exact: True  | approximate: True  | maxdiff: 0.0
dw2             | exact: True  | approximate: True  | maxdiff: 0.0
db2             | exact: True  | approximate: True  | maxdiff: 0.0
dhpreact        | exact: True  | approximate: True  | maxdiff: 0.0
dbnmeani        | exact: False | approximate: False | maxdiff: 0.004536554217338562
dbnstdi         | exact: False | approximate: